In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

In [3]:
df=pd.read_csv(r"C:\Users\SHILA\OneDrive\Desktop\PYTHON\ML_practiceDays\Datasets\Titanic-Dataset.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
460,461,1,1,"Anderson, Mr. Harry",male,48.00,0,0,19952,26.5500,E12,S
478,479,0,3,"Karlsson, Mr. Nils August",male,22.00,0,0,350060,7.5208,NaN,S
474,475,0,3,"Strandberg, Miss. Ida Sofia",female,22.00,0,0,7553,9.8375,NaN,S
305,306,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S
325,326,1,1,"Young, Miss. Marie Grice",female,36.00,0,0,PC 17760,135.6333,C32,C


In [4]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
575,0,3,male,19.0,0,0,14.5000,S
410,0,3,male,NaN,0,0,7.8958,S
220,1,3,male,16.0,0,0,8.0500,S
798,0,3,male,30.0,0,0,7.2292,C
877,0,3,male,19.0,0,0,7.8958,S


In [5]:
xtrain, xtest, ytrain, ytest=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [6]:
xtrain.sample(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
663,3,male,36.0,0,0,7.4958,S
401,3,male,26.0,0,0,8.0500,S


In [7]:
ytrain.sample(2)

565    0
878    0
Name: Survived, dtype: int64

In [8]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [9]:
# apply imputation
si_age=SimpleImputer()
si_embarked=SimpleImputer(strategy='most_frequent')

xtrain_age=si_age.fit_transform(xtrain[['Age']])
xtrain_embarked=si_embarked.fit_transform(xtrain[['Embarked']])

xtest_age=si_age.transform(xtest[['Age']])
xtest_embarked=si_embarked.transform(xtest[['Embarked']])

In [10]:
xtrain_age.shape, xtrain_embarked.shape

((712, 1), (712, 1))

In [11]:
# onehotencoder on sex and embarked, after transformed it is np arrays always
ohe_sex=OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked=OneHotEncoder(sparse_output=False, handle_unknown='ignore')

xtrain_sex=ohe_sex.fit_transform(xtrain[['Sex']])
xtrain_embarked=ohe_embarked.fit_transform(xtrain[['Embarked']])

xtest_sex=ohe_sex.transform(xtest[['Sex']])
xtest_embarked=ohe_embarked.transform(xtest[['Embarked']])

In [12]:
xtrain_sex.shape, xtrain_embarked.shape

((712, 2), (712, 4))

In [13]:
xtrain_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]], shape=(712, 2))

In [14]:
xtrain_rem=xtrain.drop(columns=['Sex','Age','Embarked'])
xtest_rem=xtest.drop(columns=['Sex','Age','Embarked'])

In [15]:
xtrain_transformed=np.concatenate((xtrain_rem,xtrain_sex,xtrain_age,xtrain_embarked), axis=1)
xtest_transformed=np.concatenate((xtest_rem,xtest_sex,xtest_age,xtest_embarked), axis=1)

In [16]:
clf=DecisionTreeClassifier()
clf.fit(xtrain_transformed,ytrain) # model training

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [17]:
y_pred=clf.predict(xtest_transformed)

In [18]:
from sklearn.metrics import accuracy_score
accuracy_score(ytest, y_pred)

0.7877094972067039

In [19]:
import pickle
pickle.dump(ohe_sex,open('ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('ohe_embarked.pkl','wb'))
pickle.dump(clf,open('clf.pkl','wb'))